In [9]:
#r "../Polars.NET.Core/bin/Debug/net10.0/Polars.NET.Core.dll"
#r "../Polars.FSharp/bin/Debug/net10.0/Polars.FSharp.dll"

open Microsoft.DotNet.Interactive.Formatting
open System
open Polars.FSharp
let display (x: obj) = x
Display.init()

In [10]:
// Define data
type WeatherData = { Date: string; City: string; Temperature: float; Rain: bool }

let data = [
    { Date="2023-01-01"; City="London";     Temperature=10.5; Rain=true }
    { Date="2023-01-01"; City="Manchester"; Temperature=9.0;  Rain=true }
    { Date="2023-01-02"; City="London";     Temperature=12.1; Rain=false }
]

let df = 
    DataFrame.ofRecords data
    |> pl.withColumn (pl.col("Date").Str.ToDate "%Y-%m-%d")

df

Datedate32,Cityutf8view,Temperaturedouble,Rainbool
2023-01-01,"""London""",10.5,true
2023-01-01,"""Manchester""",9,true
2023-01-02,"""London""",12.1,false


In [11]:
let df_converted =
    df 
    |> pl.filter (pl.col "City" .== pl.lit "London")
    |> pl.select [
        pl.col("Date")
        (pl.col("Temperature") * pl.lit 1.8 + pl.lit 32.0).Alias "Temp_F"
    ]

df_converted

Datedate32,Temp_Fdouble
2023-01-01,50.9
2023-01-02,53.78


In [12]:
let londonDf = 
    df 
    |> pl.filter (pl.col "City" .== pl.lit "London")

londonDf

Datedate32,Cityutf8view,Temperaturedouble,Rainbool
2023-01-01,"""London""",10.5,true
2023-01-02,"""London""",12.1,false


In [13]:
let aggDf = 
    df
    |> pl.groupBy [pl.col "City"] [
        pl.col("Temperature").Mean().Alias "Avg_Temp"
        pl.col("Temperature").Max().Alias "Max_Temp"
        pl.col("Rain").Sum().Alias "Rainy_Days" // bool sum -> count of true
        pl.count().Alias "Total_Records"
    ]

aggDf

Cityutf8view,Avg_Tempdouble,Max_Tempdouble,Rainy_Daysuint32,Total_Recordsuint32
"""London""",11.3,12.1,1,2
"""Manchester""",9,9,1,1


In [14]:
let windowDf = 
    df
    |> pl.select [
        pl.col "Date"
        pl.col "City"
        pl.col "Temperature"
        
        // Over(Date): Calculate mean value by group
        (pl.col("Temperature").Mean().Over(pl.col "Date"))
            .Alias "Daily_Avg"
            
        (pl.col "Temperature" - pl.col("Temperature").Mean().Over(pl.col "Date"))
            .Alias "Diff"
    ]
    |> pl.sort (pl.col "Date", false)

windowDf

Datedate32,Cityutf8view,Temperaturedouble,Daily_Avgdouble,Diffdouble
2023-01-01,"""London""",10.5,9.75,0.75
2023-01-01,"""Manchester""",9,9.75,-0.75
2023-01-02,"""London""",12.1,12.1,0


In [15]:
let lf = 
    df.Lazy()
        .Filter(pl.col "Temperature" .> pl.lit 10.0)
        .GroupBy(
            [pl.col "City"],                                         
            [pl.mean(pl.col "Temperature").Alias "Lazy_Avg_Temp"]   
        )

printfn "%s" (lf.Explain(optimized=true))

let finalDf = lf.Collect()

finalDf

AGGREGATE[maintain_order: true]
  [col("Temperature").mean().alias("Lazy_Avg_Temp")] BY [col("City")]
  FROM
  FILTER [(col("Temperature")) > (10.0)]
  FROM
    DF ["Date", "City", "Temperature", "Rain"]; PROJECT["Temperature", "City"] 2/4 COLUMNS


Cityutf8view,Lazy_Avg_Tempdouble
"""London""",11.3


In [16]:
let data = ["10.50"; "20.25"; null]
let s = Series.create("str_vals", data)

let sDec = s.Cast(DataType.Decimal(Some 10, Some 2))

let logic (opt: decimal option) =
    opt |> Option.map (fun d -> d * 2m)

let res = sDec.MapOption(logic, DataType.Decimal(Some 10, Some 2))

let res_frame = res.ToFrame()

res_frame

str_valsdecimal128
21
40.5
null
